# einops-reduce — ex6: spatial pyramid pooling with imshow per level

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-reduce`. Running the final beacon cell reports progress against the `Einops: Reduce` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Reduce` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-reduce`** (exercise 6). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-reduce"
DD_SUBTOPIC = "Einops: Reduce"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## einops.reduce — quick refresher

`reduce(tensor, pattern, op)` collapses one or more named axes with a reduction `op` ∈ `{'mean', 'sum', 'max', 'min', 'prod'}`. Drop an axis name on the right side to reduce it; keep it inside parentheses on the left and decompose first to do windowed pooling.

The exercises below stop being about *which op?* and start being about *reduce as part of a larger pipeline* — pyramid pooling, per-channel normalization, argmax-without-`torch.argmax`, top-k by repeated masked max. Each one needs visualization or print-debug to be solvable in your head.

### Exercise 6 — spatial pyramid pooling with imshow per level

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Build a spatial pyramid (1×1, 2×2, 4×4 mean pools) using einops.reduce, and visualize each pyramid level as an imshow to feel how information collapses as the grid coarsens.
> Keywords: pyramid, pooling, visualization, matplotlib
> ```

**KCs targeted:** `reduce-axis-decomposition`, `reduce-mean`

Spatial Pyramid Pooling (SPP) and the pyramid pooling module in PSPNet both produce a *multi-scale* summary of a feature map by average-pooling it to several fixed grid sizes (1×1, 2×2, 4×4, …) and concatenating the results. The visualization tells you which regions of the image survived at which scale.

Implement `ex6_pyramid_pool(feat, levels)`:
1. `feat` has shape `(C, H, W)` (single image, multi-channel).
2. `levels` is a list of grid sizes, e.g. `[1, 2, 4]`.
3. For each `L` in `levels`, mean-pool `feat` down to `(C, L, L)` using one `reduce` call with axis decomposition. Hint: `'c (h p1) (w p2) -> c h w'` with `h=L, w=L` works when `H, W` are multiples of `L`.
4. Plot the **channel-0** result at each level as imshow in a row of subplots, titled `'L=1'`, `'L=2'`, `'L=4'` etc.
5. Return a list of pooled tensors, one per level.

Looking at the plots, you should see the highest-energy regions of channel 0 survive even at the coarsest level.

In [ ]:
def ex6_pyramid_pool(feat: Tensor, levels: list[int]) -> list[Tensor]:
    import matplotlib.pyplot as plt
    C, H, W = feat.shape
    pools = []
    for L in levels:
        assert H % L == 0 and W % L == 0, f'level {L} must divide H,W'
        p = reduce(feat, 'c (h p1) (w p2) -> c h w', 'mean', h=L, w=L)
        pools.append(p)
    fig, axes = plt.subplots(1, len(levels), figsize=(3 * len(levels), 3))
    if len(levels) == 1:
        axes = [axes]
    for ax, L, p in zip(axes, levels, pools):
        ax.imshow(p[0].cpu().numpy(), cmap='viridis')
        ax.set_title(f'L={L}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    return pools


<details><summary>Solution</summary>

```python
def ex6_pyramid_pool(feat: Tensor, levels: list[int]) -> list[Tensor]:
    import matplotlib.pyplot as plt
    C, H, W = feat.shape
    pools = []
    for L in levels:
        assert H % L == 0 and W % L == 0, f'level {L} must divide H,W'
        p = reduce(feat, 'c (h p1) (w p2) -> c h w', 'mean', h=L, w=L)
        pools.append(p)
    fig, axes = plt.subplots(1, len(levels), figsize=(3 * len(levels), 3))
    if len(levels) == 1:
        axes = [axes]
    for ax, L, p in zip(axes, levels, pools):
        ax.imshow(p[0].cpu().numpy(), cmap='viridis')
        ax.set_title(f'L={L}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    return pools
```

**Why the pattern works.** `'c (h p1) (w p2) -> c h w'` with `h=L, w=L` decomposes `H = L * p1` and `W = L * p2`, then drops the `p1, p2` axes — implicitly meaning *reduce over those axes* via the `'mean'` op. One line, three jobs.

**Why visualize?** A 1×1 pool is just a number per channel — fine for a classifier head, but invisible to inspection. A 4×4 pool is an image you can *look at*. The pyramid plots make it obvious that L=1 throws away spatial structure entirely while L=4 still preserves the hot spot's location.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex6'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex6',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()